# Classical Machine Learning

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/T31SHA/data-ml-handbook/blob/main/notebooks/09_classical_ml.ipynb)

Runnable examples from **Data/ML Handbook** — section `09_classical_ml`. Read the explanations in the [docs](https://t31sha.github.io/data-ml-handbook/handbook/09_classical_ml/).

In [ ]:
# Setup: works in Colab or locally from the repo root
import os, subprocess, sys
if not os.path.exists("code") and not os.path.exists("../code"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/T31SHA/data-ml-handbook.git"], check=True)
    os.chdir("data-ml-handbook")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=False)
if os.path.exists("../code"): os.chdir("..")
os.chdir("code"); os.makedirs("data", exist_ok=True)
for prep in ["proj01_make_csv.py", "s02_load.py"]:          # shared datasets used by many snippets
    if not os.path.exists("data/housing.parquet"): subprocess.run([sys.executable, prep], check=True)
sys.path.insert(0, "../figures/src")

### `code/s07_models.py`

In [ ]:
import time, pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
import lightgbm as lgb, xgboost as xgb
X, y = load_breast_cancer(return_X_y=True)
cv = StratifiedKFold(5, shuffle=True, random_state=0)
models = {"logistic (scaled)": make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=5000)),
          "decision tree d=4": DecisionTreeClassifier(max_depth=4, random_state=0),
          "random forest": RandomForestClassifier(500, n_jobs=-1, random_state=0),
          "hist GBM": HistGradientBoostingClassifier(random_state=0),
          "LightGBM": lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, verbose=-1),
          "XGBoost": xgb.XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=4),
          "SVM RBF (scaled)": make_pipeline(StandardScaler(), SVC(C=3, gamma="scale")),
          "kNN k=7 (scaled)": make_pipeline(StandardScaler(), KNeighborsClassifier(7)),
          "Gaussian NB": GaussianNB()}
rows = []
for name, m in models.items():
    t = time.perf_counter(); s = cross_val_score(m, X, y, cv=cv, scoring="roc_auc")
    rows.append((name, s.mean(), s.std(), time.perf_counter() - t))
print(pd.DataFrame(rows, columns=["model", "ROC-AUC", "std", "sec"]).sort_values("ROC-AUC", ascending=False).round(4).to_string(index=False))

### `code/s07_unsup.py`

In [ ]:
import numpy as np
from sklearn.datasets import make_blobs, make_moons
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering, HDBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, adjusted_rand_score
X, y = make_blobs(1000, centers=4, cluster_std=1.2, random_state=0)
for k in range(2, 7):
    km = KMeans(k, n_init=10, random_state=0).fit(X)
    print(f"k={k} inertia={km.inertia_:8.0f} silhouette={silhouette_score(X, km.labels_):.3f}")
gm = [GaussianMixture(k, random_state=0).fit(X).bic(X) for k in range(2, 7)]
print("GMM BIC best k =", 2 + int(np.argmin(gm)))
Xm, ym = make_moons(600, noise=0.06, random_state=0)
for name, m in [("k-means", KMeans(2, n_init=10, random_state=0)), ("agglomerative-single", AgglomerativeClustering(2, linkage="single")),
                ("DBSCAN eps=.2", DBSCAN(eps=0.2, min_samples=5)), ("HDBSCAN", HDBSCAN(min_cluster_size=20))]:
    print(f"moons  {name:22s} ARI={adjusted_rand_score(ym, m.fit_predict(Xm)):.3f}")

### `code/s07_imbalance.py`

In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score, precision_recall_curve, f1_score, accuracy_score
X, y = make_classification(50_000, 20, n_informative=6, weights=[0.99], flip_y=0.002, random_state=0)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, stratify=y, test_size=0.3, random_state=0)
X_tr2, X_val, y_tr2, y_val = train_test_split(X_tr, y_tr, stratify=y_tr, test_size=0.3, random_state=0)
print(f"positives: {y.mean():.2%}   'always 0' accuracy = {accuracy_score(y_te, np.zeros_like(y_te)):.4f}  <- useless")
for cw in [None, "balanced"]:
    m = LogisticRegression(class_weight=cw, max_iter=2000).fit(X_tr2, y_tr2)
    pv, pt = m.predict_proba(X_val)[:, 1], m.predict_proba(X_te)[:, 1]
    P, R, T = precision_recall_curve(y_val, pv); f1 = 2 * P * R / (P + R + 1e-12); t = T[np.argmax(f1[:-1])]
    print(f"class_weight={str(cw):8s} ROC-AUC={roc_auc_score(y_te, pt):.3f} PR-AUC={average_precision_score(y_te, pt):.3f} "
          f"F1@0.5={f1_score(y_te, pt > .5):.3f}  F1@tuned({t:.2f})={f1_score(y_te, pt > t):.3f}")